In [30]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sb
import json
plt.rcParams['figure.figsize'] = [8, 5]
from sklearn.tree import DecisionTreeRegressor
from scipy.stats import norm

In [49]:
# Data generation
N = int(100000) # number of samples

p = 10 # number of covariates
sig = 5
b = 5
a = 0.5
X = np.random.normal(0,1, (N,p))
eps = np.random.normal(0,sig, N)

mu = b*(X[:, 0] <= 0)*(1 + a*(X[:,1] >0 ) + ((X[:,1]*X[:,2]) > 0) )
y = mu + eps

In [75]:
tr = DecisionTreeRegressor(random_state=0, max_depth=3, min_samples_leaf=4)
path = tr.cost_complexity_pruning_path(X, y)
ccp_alphas, impurities = path.ccp_alphas, path.impurities

for ccp_alpha in ccp_alphas[::-1]:
    tr = PTree(random_state=0, ccp_alpha=ccp_alpha, max_depth=3, min_samples_leaf=4)
    tr.fit(X, y)
    
    print("p-vals: ", tr.p_val)

p-vals:  0
p-vals:  0.0
p-vals:  0.0
p-vals:  0.2583318232223175
p-vals:  1.9139895551940955
p-vals:  3.435065953415032


In [67]:
tree = PTree(random_state=0, ccp_alpha=ccp_alpha, max_depth=3, min_samples_leaf=4)
tree.fit(X,y)

In [82]:
def nested_stopping(trees, stopping_significance):
    prev_tree = trees[0]
    for tree in trees:
        tree_significance = tree.p_val
        if tree_significance > stopping_significance:
            print("Significance of tree :", tree_significance)
            return prev_tree
        else:
            prev_tree = tree
            
    print("Significance of tree :", tree_significance)
    return prev_tree

class PTree(DecisionTreeRegressor):
    """
    ...Description of tree...

    Set random_state to ensure consistent outputs
    """
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.p_val = None

    def fit(self,X,y):
        # Fit function
        super().fit(X,y)

        # Compute total_values
        d = X.shape[1] # Features space Bonferroni Correction
        p_vals = self._node_p_vals(self)
        self.p_val = d*sum([p for p in p_vals.values()])

    def _node_p_vals(self):
        n_nodes = self.tree_.node_count
        children_left = self.tree_.children_left
        children_right = self.tree_.children_right
        feature = self.tree_.feature
        threshold = self.tree_.threshold
        values = self.tree_.value
    
        n_samples = self.tree_.n_node_samples
        impurities = self.tree_.impurity
    
        p_vals = {}
    
        node_depth = np.zeros(shape=n_nodes, dtype=np.int64)
        is_leaves = np.zeros(shape=n_nodes, dtype=bool)
        stack = [(0, 0)]  # start with the root node id (0) and its depth (0)
        while len(stack) > 0:
            # `pop` ensures each node is only visited once
            node_id, depth = stack.pop()
            node_depth[node_id] = depth
            
            # If the left and right child of a node is not the same we have a split
            # node
            is_split_node = children_left[node_id] != children_right[node_id]
            # If a split node, append left and right children and depth to `stack`
            # so we can loop through them
            if is_split_node:
                stack.append((children_left[node_id], depth + 1))
                stack.append((children_right[node_id], depth + 1))
    
                # Cal
                # Calculate p-vals for this node
                #The formula for: impurty = SSE/N
                n = n_samples[node_id]
                n_left = n_samples[children_left[node_id]]
                n_right =  n_samples[children_right[node_id]]
    
                # S 
                S = n*impurities[node_id]
                
                # S_j,theta
                S_j_l = n_left*impurities[children_left[node_id]]
                S_j_r = n_right*impurities[children_right[node_id]]
                S_j = S_j_l + S_j_r
    
                # U_j
                U_j = (S - S_j)/ (S/n)
    
                p_j = P_U(U_j, n)
                
                p_vals[node_id] = p_j
            else:
                is_leaves[node_id] = True
        return p_vals

def P_U(u,n):
	K = np.sqrt(u) - ((2*(np.log(np.log(n))))**(-1/2))*(np.log(np.log(np.log(n))) + np.log(2))
	p = (1 - norm.cdf(K)**(2*np.log(n/2)))
	return p

TypeError: 'type' object is not subscriptable

In [72]:
X.shape

(100000, 10)